# MiniMind Colab Runner

这个 Notebook **只负责运行实验**。

- 本地电脑：阅读、搜索、修改源码和写学习笔记。
- GitHub `study` 分支：在本地与 Colab 之间传递代码。
- Colab `/content/minimind`：每次运行时生成的临时代码副本，Runtime 被删除后会消失。
- Google Drive `MyDrive/minimind-runtime`：永久保存权重、checkpoint、数据和实验结果。

## Notebook 本身怎样保存

第一次打开后选择 **File → Save a copy in Drive**。以后从 Google Drive 打开这份副本。不要依赖未保存的临时 Notebook 标签页。

## 断线后怎样重新连接

1. 重新选择 GPU Runtime。
2. 从头运行到“环境验证”单元格。它会重新挂载 Drive、clone/pull 代码并恢复输出目录链接。
3. 普通 Python 变量会丢失；Drive 中的权重和 checkpoint 不会丢失。
4. 续训时在训练命令中添加 `--from_resume 1`。


## 1. 固定配置

通常只需要修改 `REPO_URL` 和 `BRANCH`。

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Napabana/minimind.git"
BRANCH = "study"
REPO_DIR = Path("/content/minimind")
DRIVE_ROOT = Path("/content/drive/MyDrive/minimind-runtime")


## 2. 挂载 Google Drive

Drive 是持久存储。`out`、`checkpoints`、大型数据集和实验结果都放这里。

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
for name in ("out", "checkpoints", "data", "results"):
    (DRIVE_ROOT / name).mkdir(parents=True, exist_ok=True)
print("持久目录：", DRIVE_ROOT)


## 3. 获取最新代码

新 Runtime 会 clone；已经存在仓库时会执行安全的 fast-forward pull。Colab 不用于修改源码，因此检测到未提交改动时会停止，防止覆盖文件。

In [ ]:
import subprocess

def run(*args, capture=False):
    return subprocess.run(args, check=True, text=True, capture_output=capture)

if (REPO_DIR / ".git").exists():
    dirty = run("git", "-C", str(REPO_DIR), "status", "--short", capture=True).stdout.strip()
    if dirty:
        raise RuntimeError("Colab 仓库存在未提交修改，请先保存，不能自动 pull：\n" + dirty)
    run("git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} 已存在但不是 Git 仓库，请检查后手动处理。")
else:
    run("git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR))

commit = run("git", "-C", str(REPO_DIR), "rev-parse", "HEAD", capture=True).stdout.strip()
print("当前 commit：", commit)


## 4. 把输出目录连接到 Drive

训练脚本仍然写入仓库里的 `out/` 和 `checkpoints/`，但实际文件会落到 Google Drive。每次新 Runtime 都要重新运行本单元格。

In [ ]:
import os

for name in ("out", "checkpoints"):
    link = REPO_DIR / name
    target = DRIVE_ROOT / name
    if link.is_symlink():
        if link.resolve() != target.resolve():
            raise RuntimeError(f"{link} 指向了意外位置：{link.resolve()}")
    elif link.exists():
        raise RuntimeError(f"{link} 已经存在。先确认其中是否有重要文件，再手动处理。")
    else:
        os.symlink(target, link, target_is_directory=True)
    print(f"{link} -> {target}")


## 5. 安装当前阶段的最小依赖

不要执行完整 `requirements.txt`。当前 Colab 保留自带的 PyTorch、CUDA 和 NumPy，只安装文本模型学习需要的包。新 Runtime 通常需要重新运行。

In [ ]:
%pip install -q "transformers==4.57.6" "datasets==3.6.0" "pyarrow==23.0.1"

## 6. 环境验证

看到 CUDA、GPU、commit 和 logits shape 后，运行环境才算准备完成。

In [ ]:
import sys
import numpy as np
import torch
import transformers
import datasets

sys.path.insert(0, str(REPO_DIR))
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = "cuda" if torch.cuda.is_available() else "cpu"
config = MiniMindConfig(hidden_size=64, num_hidden_layers=2, flash_attn=False)
model = MiniMindForCausalLM(config).to(device)
input_ids = torch.randint(0, config.vocab_size, (2, 16), device=device)
with torch.no_grad():
    output = model(input_ids)
assert output.logits.shape == (2, 16, config.vocab_size)
print("logits shape:", tuple(output.logits.shape))
print("环境验证通过 ✓")


## 7. 实验区

只在这里写调用和打印代码。需要保留的源码修改回到本地完成，然后 `git push`，再重新运行“获取最新代码”。

训练数据建议放在 `MyDrive/minimind-runtime/data/`。训练结果与断点会通过上面的目录链接自动保存到 Drive。

In [ ]:
# 示例：查看当前代码状态。这里不要编辑源码文件。
print("代码目录：", REPO_DIR)
print("数据目录：", DRIVE_ROOT / "data")
print("模型输出：", DRIVE_ROOT / "out")
print("训练断点：", DRIVE_ROOT / "checkpoints")
print("实验结果：", DRIVE_ROOT / "results")


## 8. 训练命令模板（需要时再运行）

MiniMind 脚本使用相对路径，因此从 `trainer` 目录启动。第一次训练去掉 `--from_resume 1`；断线后续训再加上。先用小数据、小模型验证，不要直接启动完整训练。

In [ ]:
%cd /content/minimind/trainer

# 首次训练示例：
# !python train_full_sft.py --data_path "/content/drive/MyDrive/minimind-runtime/data/toy_agent_sft.jsonl" --hidden_size 64 --num_hidden_layers 2 --batch_size 2 --epochs 1

# 断线后从 checkpoint 续训：
# !python train_full_sft.py --data_path "/content/drive/MyDrive/minimind-runtime/data/toy_agent_sft.jsonl" --hidden_size 64 --num_hidden_layers 2 --batch_size 2 --epochs 1 --from_resume 1


## 最终规则

- **学习代码：本地 VS Code/Cursor。** 使用跳转定义、全局搜索、断点和 Git diff。
- **修改代码：本地 `study` 分支。** 修改后 commit + push。
- **运行代码：Colab。** pull 后执行实验。
- **保存大文件：Google Drive。** 不要把权重和大数据推到 GitHub。
- **记录实验：本地 Markdown。** 每次记录 commit、配置、输入、输出、结论与边界。